# The Complete Guide to Building Your First AI Agent with DeepSeek

This notebook adapts the agent from the Medium article to use the DeepSeek LLM instead of OpenAI. We will build an agent that can:
1.  Search the web using DuckDuckGo.
2.  Get live stock prices using Yahoo Finance.

### Step 1: Install Required Libraries

In [2]:
%pip install -q langchain langchain-deepseek duckduckgo-search yfinance

Note: you may need to restart the kernel to use updated packages.


### Step 2: Imports and API Key Setup

In [4]:
pip show langchain-deepseek

Name: langchain-deepseek
Version: 0.1.3
Summary: An integration package connecting DeepSeek and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/nhujaw/miniconda3/lib/python3.13/site-packages
Requires: langchain-core, langchain-openai
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, DuckDuckGoSearchRun

# <<< MODIFIED: Import ChatDeepseek instead of ChatOpenAI
from langchain_deepseek import ChatDeepseek 

ImportError: cannot import name 'ChatDeepseek' from 'langchain_deepseek' (/home/nhujaw/miniconda3/lib/python3.13/site-packages/langchain_deepseek/__init__.py)

In [ ]:
# --- API Key Configuration ---
# For security, it's best to set this as an environment variable.
# However, for this notebook, you can paste it directly.

# os.environ["DEEPSEEK_API_KEY"] = "YOUR_DEEPSEEK_API_KEY"

# <<< ⚠️ PASTE YOUR DEEPSEEK API KEY HERE
DEEPSEEK_API_KEY = "YOUR_DEEPSEEK_API_KEY"

### Step 3: Define the Agent's Tools

Tools are the functions the agent can decide to call. We'll create two:
1. A pre-built tool for web search.
2. A custom tool for getting stock prices.

In [ ]:
# Tool 1: Web Search (pre-built)
search_tool = DuckDuckGoSearchRun()

In [ ]:
# Tool 2: Get Stock Price (custom)
@tool
def get_stock_price(symbol: str) -> float:
    """Returns the latest stock price for a given ticker symbol."""
    import yfinance as yf
    try:
        ticker = yf.Ticker(symbol)
        price = ticker.info.get('regularMarketPrice')
        if price is not None:
            return price
        else:
            # Fallback for pre-market/post-market or if regularMarketPrice is missing
            hist = ticker.history(period="1d")
            if not hist.empty:
                return hist['Close'].iloc[-1]
            return f"Price for symbol {symbol} not found."
    except Exception as e:
        return f"An error occurred for symbol {symbol}: {e}"

In [ ]:
# Create a list of all tools the agent can use
tools = [search_tool, get_stock_price]

### Step 4: Initialize the LLM and Build the Agent

In [ ]:
# <<< MODIFIED: Initialize the DeepSeek model
llm = ChatDeepseek(
    model="deepseek-chat", 
    temperature=0,
    api_key=DEEPSEEK_API_KEY
)

# Create the prompt template. This tells the agent how to behave.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# Create the agent by binding the LLM with the tools and prompt
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the Agent Executor, which runs the agent's reasoning loop
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

### Step 5: Run the Agent!

Now we can ask the agent questions. The `verbose=True` setting will show the agent's thought process.

#### Example 1: Using the Stock Price Tool

In [ ]:
response1 = agent_executor.invoke({"input": "What is the stock price of Apple (AAPL)?"})

In [ ]:
print("\nFinal Answer:", response1['output'])

#### Example 2: Using Both Search and Stock Price Tools

In [ ]:
response2 = agent_executor.invoke({"input": "What was the latest news about NVIDIA and what is their current stock price (NVDA)?"})

In [ ]:
print("\nFinal Answer:", response2['output'])